## Part 1 - Downloading and Preparing the Receptor for AutodDock

In [1]:
#This script will create a folder called "single-docking" for our experiment
#Then, it will print all "ATOM" and "TER" lines from a given PDB into a new file

#Let's make a folder first. We need to import the os and path library
import os
from pathlib import Path

#Then, we define the path of the folder we want to create.
singlepath = Path('./virtual_screening/')

#Now, we create the folder using the os.mkdir() command
#The if conditional is just to check whether the folder already exists
if os.path.exists(singlepath):
    print('path already exists')
if not os.path.exists(singlepath):
    os.mkdir(singlepath)
    print('path was successfully created')

#Now we assign a variable "protein" with the name and extension of our pdb
protein = '8vb5.pdb'

#And we use the following script to selectively print the lines that contain the
#string "ATOM" and "TER" into a new file inside our recently created folder
with open(singlepath / '8vb5_prot.pdb', 'w') as g:
    f = open(protein, 'r')
    for line in f:
        row = line.split()
        if row[0] == 'ATOM' and line[21] == 'A':
            g.write(line)
        elif row[0] == 'TER':
            g.write('TER\n')
    g.write('END')
    print('file successfully created')

path already exists
file successfully created


Add the polar hydrogens of your protein and parameterize it based on the pKa of each aminoacid at pH 7.4 with the **AMBER99ff** force field using **pdb2pqr**, followed by deletion of non-polar hydrogens and conversion into **PDBQT** file using **MGLtools**.



In this case, pdb2pqr generates an intermediate **PQR** file, a modification of the PDB format which allows users to add charge and radius parameters to existing PDB data. This information is then unaltered during the use of **MGLtools**.

In [5]:
#First, using pdb2pqr to parameterize our receptor with AMBER99ff, maintaining
#the chain IDs and setting up the receptor at a pH of 7.4
!/home/ssm-user/miniforge3/envs/docking/bin/pdb2pqr30 --ff AMBER --keep-chain --titration-state-method propka --with-ph 7.4 $singlepath/8vb5_prot.pdb $singlepath/8vb5_prot.pqr

#Then, convert the .pqr file into a .pdbqt file while deleting non-polar
#hydrogens but without changing the AMBER parameters added to the protein
!/home/ssm-user/apps/mgltools/bin/pythonsh /home/ssm-user/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_receptor4.py \
-r ./virtual_screening/8vb5_prot.pqr \
-o ./virtual_screening/8vb5_prot.pdbqt \
-C \
-U nphs_lps \
-v

INFO:PDB2PQR v3.7.1: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: virtual_screening/8vb5_prot.pdb
INFO:Setting up molecule.
INFO:Created biomolecule object with 289 residues and 2306 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:Attempting to repair 1 missing atoms in biomolecule.
INFO:Added atom OXT to residue GLN A 990 at coordinates -5.748, 17.260, 14.414
INFO:Updating disulfide bridges.
INFO:Debumping biomolecule.
INFO:Assigning titration states with PROPKA.
INFO:
propka3.5.1         

In [ ]:
# 옵션 설명:
# -C : unstandard residue check but continue
# -U nphs_lps : non-polar hydrogens 합침 + lone pairs 추가
# -v : verbose

You are all set with your target protein!

## Part 2 – Downloading and Preparing the Ligand for AutoDock

In [6]:
#Here, we will be extracting Indinavir, which is present in the structure of
#HIV-2 protease (yes! this is a simulation with experimental validation!)
#The approach is similar to printing the ATOM and TER lines, but we are using
#the residue name given to the ligand in the experimentally solved structure: MK1
protein = "8vb5.pdb"

with open(singlepath/"AAC.pdb","w") as g:
  f = open(protein,'r')
  for line in f:
    row = line.split()
    # extract ligand
    if line.startswith('HETATM') and line[17:20]=="AAC" and line[21]=='A':
      g.write(line)
  g.write("END")

In [7]:
import numpy as np
with open('8vb5.pdb', 'r') as f:
  ligand_geom = []
  for l in f:
    if l.startswith('HETATM') and l[17:20]=='AAC' and l[21]=='A':
      x, y, z = float(l[30:38]), float(l[38:46]), float(l[46:54])
      #print(x,y,z)
      ligand_geom.append([x,y,z])
  #print("Ligand Geom:\n", ligand_geom)

  ligand_geom = np.array(ligand_geom)
  ligand_center = ligand_geom.mean(axis=0)
  print(f"\n Geometric Center of a ligand: {ligand_center[0]:.3f} {ligand_center[1]:.3f} {ligand_center[2]:.3f}")


 Geometric Center of a ligand: -1.128 6.733 -18.115


In [8]:
# Convert pdb file -> mol2 file
# Then, prepare ligand for docking using the Autodock script
!obabel -ipdb $singlepath/AAC.pdb -omol2 -O AAC.mol2

!~/apps/mgltools/bin/pythonsh ~/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py \
    -l AAC.mol2 \
    -o {singlepath}/AAC.pdbqt \
    -U nphs_lps \
    -v

1 molecule converted
setting PYTHONHOME environment
set verbose to  True
read  AAC.mol2
setting up LPO with mode= automatic and outputfilename=  virtual_screening/AAC.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates


## Docking to ENAMINE compounds

In [10]:
import pandas as pd

df = pd.read_csv("smiles_batch_3.csv")

### 데이터프레임에서 smiles 추출해서 각각 파일로 저장

In [12]:
for i, row in df.iterrows():
    name = row["Catalog ID"]
    smiles = row["SMILES"]

    with open(f"./enamine_hits/{name}.smi", "w") as f:
        f.write(smiles)

### Convert each smi file into a pdbqt file

In [13]:
!mkdir enamine_hits_pdbqt

In [ ]:
%%bash
for smi in $(ls ./enamine_hits/*.smi)
do
 # echo $smi
 base=$(basename "$smi" .smi)
 # echo $base
 # echo "obabel -ismi $smi -opdbqt -O ./enamine_hits_pdbqt/$base.pdbqt --gen3d --fast -p 7.4 --canonical"
 obabel -ismi $smi -opdbqt -O ./enamine_hits_pdbqt/$base.pdbqt --gen3d --fast -p 7.4 --canonical
done

1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule co

In [31]:
%%bash
# smi 파일 이름(확장자 제거)
ls ./enamine_hits/*.smi | xargs -n1 basename | sed 's/.smi$//' | sort > smi_list.txt

# pdbqt 파일 이름(확장자 제거)
ls ./enamine_hits_pdbqt/*.pdbqt | xargs -n1 basename | sed 's/.pdbqt$//' | sort > pdbqt_list.txt

# 두 리스트 비교
diff smi_list.txt pdbqt_list.txt


### Making grid box
--------
all possible atom types should be specified for virtual screening since we will handle diverse molecules.

AutoDock Atom types:

https://autodock.scripps.edu/wp-content/uploads/sites/31/2019/03/AD4.1_bound.dat

In [32]:
%%bash
cd virtual_screening
~/apps/mgltools/bin/pythonsh ~/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_gpf4.py \
    -l AAC.pdbqt \
    -r 8vb5_prot.pdbqt \
    -y \
    -p ligand_types='H,HD,A,C,N,NA,NS,OA,OS,F,Cl,Br,S,SA' \
    -p npts='80,80,80' 

# 생성된 GPF 파일 확인
#cat 8vb5_prot.gpf
# -r: receptor (수용체) 파일
# -l: ligand (리간드) 파일 (참조용)
# -y: 자동으로 'yes' 응답
# -p ligand_types: 리간드 원자 타입들
# -p npts: 그리드 포인트 수 (x,y,z)
cd ..

setting PYTHONHOME environment
setting ligand_types: newvalue= H HD A C N NA NS OA OS F Cl Br S SA


## Run Autogrid command
-------

fld: field file
그리드 크기와 좌표 정보
사용 가능한 MAP 파일들의 목록
그리드 중심점, 간격 등 설정 정보

map: Atom-specific Maps
각 원자 타입별 실제 에너지 데이터
3D 격자의 각 점에서의 상호작용 에너지
특정 원자 타입이 그 위치에 있을 때의 에너지 값

In [33]:
%%bash
cd virtual_screening

# autogrid4 실행 (GPF → MAP 파일들 생성)
/home/ssm-user/miniforge3/envs/docking/bin/autogrid4 -p 8vb5_prot.gpf -l 8vb5_prot.glg

# 생성된 파일들 확인
ls -la *.fld *.map

cd ../

-rw-r--r-- 1 ssm-user ssm-user 4053383 Aug 19 01:05 8vb5_prot.A.map
-rw-r--r-- 1 ssm-user ssm-user 4200779 Aug 19 01:05 8vb5_prot.Br.map
-rw-r--r-- 1 ssm-user ssm-user 4063801 Aug 19 01:05 8vb5_prot.C.map
-rw-r--r-- 1 ssm-user ssm-user 4128891 Aug 19 01:05 8vb5_prot.Cl.map
-rw-r--r-- 1 ssm-user ssm-user 3870018 Aug 19 01:05 8vb5_prot.F.map
-rw-r--r-- 1 ssm-user ssm-user 3615127 Aug 19 01:05 8vb5_prot.H.map
-rw-r--r-- 1 ssm-user ssm-user 3652024 Aug 19 01:05 8vb5_prot.HD.map
-rw-r--r-- 1 ssm-user ssm-user 3989394 Aug 19 01:05 8vb5_prot.N.map
-rw-r--r-- 1 ssm-user ssm-user 3993685 Aug 19 01:05 8vb5_prot.NA.map
-rw-r--r-- 1 ssm-user ssm-user 3993780 Aug 19 01:05 8vb5_prot.NS.map
-rw-r--r-- 1 ssm-user ssm-user 3973907 Aug 19 01:05 8vb5_prot.OA.map
-rw-r--r-- 1 ssm-user ssm-user 3973907 Aug 19 01:05 8vb5_prot.OS.map
-rw-r--r-- 1 ssm-user ssm-user 4094462 Aug 19 01:05 8vb5_prot.S.map
-rw-r--r-- 1 ssm-user ssm-user 4104094 Aug 19 01:05 8vb5_prot.SA.map
-rw-r--r-- 1 ssm-user ssm-user 3168129 A

## Making a batch file
---------
batch file contains the list of receptors and ligands

For full list of options of AutoDock-GPU:

https://github.com/ccsb-scripps/AutoDock-GPU

Example command

```
./bin/autodock_<type>_<N>wi \
--ffile <protein>.maps.fld \
--lfile <ligand>.pdbqt \
--nrun <nruns>
```

To run multiple ligands, **a batch file** can be used.
The batch file is a text file containing the parameters to --ffile, --lfile, and --resnam each on an individual line. It is possible to only use one line to specify the Protein grid map file which means it will be used for all ligands. Here is an example:



```
./receptor1.maps.fld
./ligand1.pdbqt
Ligand 1
./receptor2.maps.fld
./ligand2.pdbqt
Ligand 2
./receptor3.maps.fld
./ligand3.pdbqt
Ligand 3
```


In [34]:
import os
from glob import glob

with open('./virtual_screening/ligand_list.txt', 'w') as fout:
    # write a header line 
    fout.write('./8vb5_prot.maps.fld \n') 
    
    # grap all ligand pdbqt files.
    files = glob('./enamine_hits_pdbqt/*.pdbqt')
    
    # iterate over all pdbqt files
    for filename in files:
        fout.write(os.path.abspath(filename) + '\n')
        fout.write(filename.split('/')[-1].split('.')[0] + '\n')

### Run AutoDock-GPU

In [ ]:
%%bash
cd virtual_screening

# 이전 결과 파일 정리
rm -f *.dlg *.xml

# AutoDock-GPU 실행 
~/apps/AutoDock-GPU/bin/autodock_gpu_128wi -B ./ligand_list.txt | tee gpu_out

# -B: 여러 리간드 한 번에 처리
# tee gpu_out: 출력을 화면과, gpu_out에 동시에 저장

AutoDock-GPU version: v1.6-7-ga46ab564d2ac5f1a1523f65239b43505a1c29364-dirty

Using 8 OpenMP threads

Running 20032 docking calculations

Cuda device:                              Tesla T4
Available memory on device:               14254 MB (total: 14913 MB)

CUDA Setup time 0.208448s
(Thread 6 is setting up Job #1)
(Thread 2 is setting up Job #2)
(Thread 7 is setting up Job #3)
(Thread 1 is setting up Job #5)
(Thread 5 is setting up Job #6)
(Thread 0 is setting up Job #7)
(Thread 3 is setting up Job #8)
(Thread 4 is setting up Job #4)

Running Job #1:
    Device: Tesla T4
    Grid map file: ./8vb5_prot.maps.fld
    Ligand file: /home/ssm-user/practice/autodock_gpu/enamine_hits_pdbqt/Z383153422.pdbqt
    Output file: Z383153422.dlg (+ xml)
    Using heuristics: (capped) number of evaluations set to 1132076
    Local-search chosen method is: ADADELTA (ad)

Rest of Setup time 0.057734s

Executing docking runs, stopping automatically after either reaching 0.15 kcal/mol standard deviation o

### Analyze gpu_out
----------------------------

In [ ]:
def get_min_affinity(dlg_file):
    """
    Extract minimum binding affinity from dlg file
    
    Args:
        dlg_file (str): Path to .dlg file
        
    Returns:
        float: Minimum binding affinity
    """
    with open(dlg_file, 'r') as f:
        lines = f.readlines()

    affinity_list = []
    for line in lines:
        if 'Estimated Free Energy of Binding' in line:
            try:
                affinity = float(line.strip().split()[-3])
                affinity_list.append(affinity)
            except (ValueError, IndexError):
                continue  # 파싱 실패 시 스킵
    
    if affinity_list:
        return min(affinity_list)
    else:
        return None  # 에너지를 찾지 못한 경우

In [ ]:
!wget --progress=bar -O drugs.txt "https://www.dropbox.com/scl/fi/d7gfdvnhbgkitwrnq1hnr/drugs.txt?rlkey=wl1dr329po25jqcsa9y3gvtkt&dl=1"

Split each line of drugs.txt into each smiles file

한 줄의 SMILES를 하나의 파일로 저장함.

In [ ]:
if not os.path.exists("drugs"):
  os.mkdir("drugs")

name2smi = {}

data = open('drugs.txt').readlines()[1:]
for dat in data[1:]:
    dat = dat.strip().split()
    if len(dat)<3:
        continue
    else:
        name, smi = dat[0], dat[2]
        name2smi[name] = smi
        if smi == 'FALSE':
            continue
        with open(f"./drugs/{name}.smi", 'w') as fout:
            fout.write(smi+'\n')

Convert each smi file into a pdbqt file.

For detailed options of openbabel, please refer to

https://open-babel.readthedocs.io/en/latest/Command-line_tools/babel.html

Openbabel을 이용해서 하나의 smiles를 하나의 pdbqt 파일로 변환함.

각 옵션에 대한 자세한 설명은 아래 문서를 참고.

https://open-babel.readthedocs.io/en/latest/Command-line_tools/babel.html

In [ ]:
!mkdir drug_pdbqt

Converting SMILES to 3D STRUCTURE (SDF) and SDF to PDBQT using RDKIT and MEEKO instead of OBABEL by `make_ligands.sh`

### Making grid box
--------
all possible atom types should be specified for virtual screening since we will handle diverse molecules.

AutoDock Atom types:

https://autodock.scripps.edu/wp-content/uploads/sites/31/2019/03/AD4.1_bound.dat

In [ ]:
%%bash
cd virtual_screening
~/apps/mgltools/bin/pythonsh ~/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_gpf4.py \
    -l 36h.pdbqt \
    -r 7f7w_prot.pdbqt \
    -y \
    -p ligand_types='H,HD,A,C,N,NA,NS,OA,OS,F,Cl,Br,S,SA' \
    -p npts='80,80,80' 

# 생성된 GPF 파일 확인
#cat 7f7w_prot.gpf
# -r: receptor (수용체) 파일
# -l: ligand (리간드) 파일 (참조용)
# -y: 자동으로 'yes' 응답
# -p ligand_types: 리간드 원자 타입들
# -p npts: 그리드 포인트 수 (x,y,z)
cd ..

In [ ]:
%%bash
cd ~/practice/ad_gpu/virtual_screening

# autogrid4 실행 (GPF → MAP 파일들 생성)
/home/ssm-user/miniforge3/envs/docking/bin/autogrid4 -p 7f7w_prot.gpf -l 7f7w_prot.glg

# 생성된 파일들 확인
ls -la *.fld *.map

cd ../

fld: field file
그리드 크기와 좌표 정보
사용 가능한 MAP 파일들의 목록
그리드 중심점, 간격 등 설정 정보

map: Atom-specific Maps
각 원자 타입별 실제 에너지 데이터
3D 격자의 각 점에서의 상호작용 에너지
특정 원자 타입이 그 위치에 있을 때의 에너지 값

In [ ]:
%%bash
cd ~/practice/ad_gpu/virtual_screening

# 이전 결과 파일 정리
rm -f *.dlg *.xml

# AutoDock-GPU 실행 
~/apps/AutoDock-GPU/bin/autodock_gpu_128wi -B ./ligand_list.txt | tee gpu_out

# -B: 여러 리간드 한 번에 처리
# tee gpu_out: 출력을 화면과, gpu_out에 동시에 저장

### Analyze gpu_out
----------------------------

In [1]:
def get_min_affinity(dlg_file):
    """
    Extract minimum binding affinity from dlg file
    
    Args:
        dlg_file (str): Path to .dlg file
        
    Returns:
        float: Minimum binding affinity
    """
    with open(dlg_file, 'r') as f:
        lines = f.readlines()

    affinity_list = []
    for line in lines:
        if 'Estimated Free Energy of Binding' in line:
            try:
                affinity = float(line.strip().split()[-3])
                affinity_list.append(affinity)
            except (ValueError, IndexError):
                continue  # 파싱 실패 시 스킵
    
    if affinity_list:
        return min(affinity_list)
    else:
        return None  # 에너지를 찾지 못한 경우

In [2]:
import glob
import os

dlg_files = glob.glob('./virtual_screening/*.dlg') 
lig_and_energy = []

print("Processing dlg files...")
for f in dlg_files:
    min_e = get_min_affinity(f)
    if min_e is not None: 
        lig_name = os.path.basename(f).split('.')[0]  
        lig_and_energy.append((lig_name, min_e))

# 결합 에너지 순으로 정렬 (가장 음수 = 가장 강한 결합)
lig_and_energy.sort(key=lambda x: x[1])

print(f"\n=== TOP LIGANDS (Total: {len(lig_and_energy)}) ===")
print(f"{'Rank':<5} {'Ligand':<25} {'Binding Affinity (kcal/mol)':<25}")
print("-" * 60)

for rank, (ligand, energy) in enumerate(lig_and_energy, 1):
    print(f"{rank:<5} {ligand:<25} {energy:<25.2f}")

# Top 10만 보기
print(f"\n=== TOP 10 ===")
for rank, (ligand, energy) in enumerate(lig_and_energy[:10], 1):
    print(f"{rank}. {ligand}: {energy:.2f} kcal/mol")

Processing dlg files...

=== TOP LIGANDS (Total: 20023) ===
Rank  Ligand                    Binding Affinity (kcal/mol)
------------------------------------------------------------
1     Z15623383                 -11.58                   
2     Z29999382                 -11.46                   
3     Z89976164                 -11.45                   
4     Z4462796186               -11.38                   
5     Z599743450                -11.33                   
6     Z227926890                -11.32                   
7     Z1213673068               -11.28                   
8     Z1262049531               -11.25                   
9     Z195595796                -11.20                   
10    Z28179136                 -11.17                   
11    Z1797379077               -11.02                   
12    Z237915472                -10.97                   
13    Z2218571548               -10.94                   
14    Z32463745                 -10.93                   
15    Z

In [7]:
import pandas as pd
name2smi = {}
try:
    # with open('./smiles_batch_3.csv', 'r') as f:
    #     for line in f:
    #         parts = line.strip().split('\t')
    #         if len(parts) >= 3:
    #             name2smi[parts[0]] = parts[2]
    df = pd.read_csv("smiles_batch_3.csv")
    for i, row in df.iterrows():
        name = row["Catalog ID"]
        smiles = row["SMILES"]
        name2smi[name] = smiles

except FileNotFoundError:
    print("SMILES file not found, proceeding without SMILES")
    name2smi = {}

# 그 다음 원래 코드 실행
with open('./virtual_screening/ligand_and_affinity.txt', 'w') as fout:
    fout.write('Rank\tLigand\tSMILES\tBinding_Affinity(kcal/mol)\n')
    
    for rank, (lig, ene) in enumerate(lig_and_energy, 1):
        smiles = name2smi.get(lig, 'N/A')  # SMILES가 없으면 'N/A'
        fout.write(f'{rank}\t{lig}\t{smiles}\t{ene:10.2f}\n')

In [8]:
!cat ./virtual_screening/ligand_and_affinity.txt

Rank	Ligand	SMILES	Binding_Affinity(kcal/mol)
1	Z15623383	O=C(COC(=O)C=1C=CC=C(C1)N2C(=O)C=3C=CC=CC3C2=O)NC(=O)NC4CCCCC4	    -11.58
2	Z29999382	O=C(NC=1C=CC=C(C1)S(=O)(=O)N2CCOCC2)C=3C=CC(=CC3)N4CCCC4=O	    -11.46
3	Z89976164	CC1(CC=2C=CC=3OCOC3C2)NC(=O)N(CC=4C=C(Cl)C=5OCCOC5C4)C1=O	    -11.45
4	Z4462796186	O=C(CCC1NC(=O)N(C1=O)C=2C=CC(Cl)=C(Cl)C2)N3CCN4C(=O)NC(=O)C4C3	    -11.38
5	Z599743450	CN1CCCN(CC1)C=2C=CC(=CN2)CNC(=O)C3=CC4=CC(=CC=C4S3)[N+]([O-])=O	    -11.33
6	Z227926890	CC=1C=C(C=C(C1C)S(=O)(=O)N2CCC(CC2)NS(=O)(=O)C=3C=CC=CC3)[N+]([O-])=O	    -11.32
7	Z1213673068	CC=1C=C(O)C(=CC1C)C=2C=C(NN2)C(=O)NCCC=3C=CC(=CC3)S(N)(=O)=O	    -11.28
8	Z1262049531	CC1CN(CC(C)O1)C(=O)CN2C=C(C=N2)NC(=O)C3=CC=C(C=N3)C(N)=O	    -11.25
9	Z195595796	CN(CC=1N=C(N)C2=C(N1)SC=3CCCCCC23)CC4=NC=5C=CC=CC5C(=O)N4	    -11.20
10	Z28179136	CC1CCCCC21NC(=O)N(CC(=O)NC3=NC4=CC=C(C=C4S3)S(C)(=O)=O)C2=O	    -11.17
11	Z1797379077	CC1=NC=CN1CC=2C=CC=C(C2)C3=NOC(=N3)C4=CC=C(C(F)=C4)S(N)(=O)=O	    -11.02
12	Z237915472

In [ ]:
!obabel -ad -ipdbqt ./virtual_screening/Acetohexamide.dlg  -opdbqt -O ./virtual_screening/Acetohexamide.docked.pdbqt